# BIY metric bank

This notebook loads the metric bank, shows how the inventory splits by category and by selector priority, runs the five worked client selections, and computes a few formulas locally. Nothing here calls a model.

Open it from the `metric-bank` directory, or from `notebooks/`. The first cell finds `src`.


In [1]:
from pathlib import Path
import sys

def find_src() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "metric_bank").is_dir():
            return candidate / "src"
    raise RuntimeError("Run this notebook from the metric-bank directory.")

SRC = find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from metric_bank.build import build_bank, load_bank
from collections import Counter

bank = load_bank()
built = build_bank()
assert bank == built, "data/metric-bank.json is stale; rebuild it"
metrics = bank["metrics"]
print(f"{len(metrics)} metrics, {len(bank['categories'])} categories, {len(bank['priors'])} priors")
print("Priorities:", dict(Counter(m["selector_priority"] for m in metrics)))
print("Catalogue matches:", sum(m["catalog_match"] for m in metrics))
print(bank["design_notes"])


159 metrics, 22 categories, 17 priors
Priorities: {'supporting': 77, 'core': 53, 'diagnostic': 24, 'avoid_as_primary': 5}
Catalogue matches: 40
The bank has one row per measurement a client benchmark can actually run. Public exams (MMLU, BIG-bench, GSM8K, HELM scenarios, Arena Elo) are priors, not metrics. Where Ragas and DeepEval name the same construct, one row is kept and the other name is an alias. Metrics marked avoid_as_primary exist so a selector can refuse BLEU, ROUGE, BERTScore, fluency, or embedding cosine as the sole quality score. Operational metrics are measured on the same run as quality. Only utility_axis=quality scores enter the 0–1 quality average, and they must already be higher-better. Lower-better risk rates are gates. A client scorecard should contain the always-on operational metrics, one speed metric chosen from the use-case kind, one to five quality metrics, and zero to five risk gates. Document vision is included for pages and scans: character and word error, p

## Overview

Counts come from `data/metric-bank.json` when that file is present, and the next cell checks it still matches the builder.


In [2]:
from collections import Counter

counts = Counter(m["category"] for m in metrics)
core = Counter(m["category"] for m in metrics if m["selector_priority"] == "core")
names = {c["id"]: c["name"] for c in bank["categories"]}
width = max(counts.values())
print(f"{'category':<28} {'rows':>5} {'core':>5}  ")
for category in bank["categories"]:
    cid = category["id"]
    bar = "#" * counts[cid]
    print(f"{cid:<28} {counts[cid]:>5} {core[cid]:>5}  {bar}")
    print(f"  {names[cid]}")
    print(f"  {category['selector_notes']}")


category                      rows  core  
accuracy                        12     4  ############
  Accuracy and classification
  Prefer these over a judge whenever a gold string, label, or list exists. Lead classification with F1 unless the client named precision or recall.
reasoning                        4     2  ####
  Reasoning
  Score the final answer deterministically. Add step correctness only when a person will read the work.
knowledge                        5     2  #####
  Knowledge and factuality
  If the product must stay inside a retrieved document, use faithfulness instead of a knowledge metric.
rag_grounding                   11     5  ###########
  RAG and grounding
  Faithfulness and answer relevancy grade the generator. Context metrics grade retrieval. Do not grade retrieval when context is frozen.
summarization                    8     2  ########
  Summarization
  Do not select ROUGE or BERTScore as the quality metric. Use coverage plus faithfulness.
code          

## Category breakdown


In [3]:
print("Quality-axis metrics by category")
for category in bank["categories"]:
    chosen = [
        m["id"] for m in metrics
        if m["category"] == category["id"] and m["utility_axis"] == "quality" and m["selector_priority"] == "core"
    ]
    if chosen:
        print(f"  {category['id']}: {', '.join(chosen)}")

print("\nRefused as a primary quality score")
for metric in metrics:
    if metric["selector_priority"] == "avoid_as_primary":
        print(f"  {metric['id']}: {metric['when_not_to_use'][0]}")


Quality-axis metrics by category
  accuracy: classification_f1, exact_match, set_f1, token_f1
  reasoning: math_equivalence, step_correctness
  knowledge: factual_correctness, unknown_abstention
  rag_grounding: answer_relevancy, context_precision, context_recall, context_relevancy, faithfulness
  summarization: summary_coherence, summary_coverage
  code: code_tests_pass
  conversation: conversation_completeness, knowledge_retention
  instruction_following: instruction_following
  structured_output: field_accuracy, json_schema_valid, sql_execution_accuracy
  vision: bbox_f1, bbox_iou, character_accuracy
  agents_tools: agent_task_success, tool_call_correct, tool_choice_correct
  multilingual: language_is, translation_comet
  robustness: long_context_accuracy
  business_ux: tone_brand_fit
  judging: g_eval, pairwise_vs_baseline, rubric_score

Refused as a primary quality score
  semantic_similarity: You need to know whether facts are right. Similar embeddings can share topic and still c

## What the selector is allowed to average

Quality-axis metrics are higher-better on 0–1. Avoid-as-primary rows are in the bank so they can be refused.


In [4]:
from metric_bank.selector import EXAMPLES, select

for example in EXAMPLES:
    card = select(example, bank)
    print("=" * 72)
    print(card["title"], f"({card['kind']}, {card['industry']})")
    print(card["narrative"])
    print("quality:    ", ", ".join(card["quality"]))
    if card["quality_alternates"]:
        print("alternates: ", ", ".join(card["quality_alternates"]))
    speed = card["speed"]["ranking"] or "none"
    extra = ", ".join(card["speed"]["also_report"])
    print("speed:      ", speed + (f"  also {extra}" if extra else ""))
    print("risk gates: ", ", ".join(f"{g['id']} {g['threshold']}" for g in card["risk_gates"]) or "none")
    print("task gates: ", ", ".join(f"{g['id']} {g['threshold']}" for g in card["task_gates"]))
    reported = card["reliability_extra"] + card["also_report"]
    if reported:
        print("reported:   ", ", ".join(reported))
    for note in card["notes"]:
        print("note:       ", note)


Health insurer claim extraction (extraction, insurance)
A claims platform extracts a fixed JSON object from an explanation-of-benefits packet: member id, service date, billed amount, allowed amount, and denial code. The caller waits for the object. Amounts may be off by one cent. The packet contains PHI.
quality:     field_accuracy, numeric_tolerance_accuracy, json_schema_valid
speed:       e2e_ms
risk gates:  pii_leakage_rate <= 0
task gates:  error_rate <= 0.01, truncation_rate <= 0.01, json_schema_valid >= 0.99
Fintech support chat (chat, finance)
A streaming support assistant answers account questions in the company's voice. A help-center knowledge base is retrieved when the agent decides it is relevant, but the product is not a system of record. Candidates include reasoning models. The assistant must refuse crimes and scams, must not refuse ordinary balance questions, and must not give personalized financial advice.
quality:     faithfulness, tone_brand_fit, answer_relevancy
speed

## Five client scorecards

The reference selector is deterministic. These are the same cards as `selection/examples.json` and the guide.


In [5]:
from metric_bank.illustrative import (
    ambiguous_context_bias,
    bbox_iou,
    case_cost_usd,
    character_accuracy,
    character_error_rate,
    expected_calibration_error,
    faithfulness_from_verdicts,
    json_schema_valid,
    pass_at_k,
    pass_hat_k,
    percentile,
    rouge_l,
    target_score,
    token_f1,
)

print("token F1", round(token_f1("the cat sat on the mat", "cat on mat"), 3))
schema = {"type": "object", "required": ["billed_amount"], "properties": {"billed_amount": {"type": "number"}}}
print("schema ok", json_schema_valid({"billed_amount": 120.5}, schema))
print("schema bad", json_schema_valid({"billed_amount": "120.50"}, schema))
print("faithfulness", faithfulness_from_verdicts([True, True, True, False]))
prices = {"input": 3.0, "cached": 0.3, "output": 15.0}
print("cost usd", case_cost_usd(1_000_000, 200_000, 100_000, prices))
print("ECE", expected_calibration_error([0.9, 0.9, 0.2, 0.2], [True, False, False, True], n_bins=5))
print("pass@2", round(pass_at_k(n=10, c=4, k=2), 3))
print("pass^2", round(pass_hat_k(n=10, c=4, k=2), 3))
print("p95 of e2e ms", percentile([80, 90, 100, 110, 400], 95))
print("target score at 2x", round(target_score(200, 100, 1000), 3))
print("ROUGE-L, for contrast only", round(rouge_l("the claim was denied", "the claim was paid"), 3))
print("BBQ-style bias", round(ambiguous_context_bias(accuracy=0.4, stereotype_aligned=7, non_unknown=10), 3))
print("CER", round(character_error_rate("120.50", "120.00"), 3), "accuracy", round(character_accuracy("120.50", "120.00"), 3))
print("box IoU", round(bbox_iou((0, 0, 10, 10), (5, 5, 15, 15)), 3))


token F1 0.667
schema ok True
schema bad False
faithfulness 0.75
cost usd 3.96
ECE 0.35
pass@2 0.667
pass^2 0.133
p95 of e2e ms 341.99999999999994
target score at 2x 0.699
ROUGE-L, for contrast only 0.75
BBQ-style bias 0.24
CER 0.167 accuracy 0.833
box IoU 0.143


## Formula sketches

Token F1, a schema check, faithfulness from claim verdicts, cost, calibration, pass@k, pass^k, and the product's log target for speed and cost.


In [6]:
print(f"{'prior':<28} {'kind':<22} citation")
for prior in bank["priors"]:
    print(f"{prior['id']:<28} {prior['kind']:<22} {prior['citation']}")
print()
print("Example of a prior boundary:")
helm = next(item for item in bank["priors"] if item["id"] == "helm")
print(helm["name"])
print("useful as:", helm["useful_as"])
print("not as:   ", helm["do_not_use_as"])


prior                        kind                   citation
mmlu                         knowledge_exam         Hendrycks et al., 2020. MMLU-Pro is the harder follow-up.
big_bench                    capability_suite       Srivastava et al., 2022. Suzgun et al., 2022 for BBH.
helm                         holistic_framework     Liang et al., 2022, Holistic Evaluation of Language Models.
gsm8k_math                   reasoning_exam         Cobbe et al., 2021. Hendrycks et al., 2021.
gpqa                         knowledge_exam         Rein et al., 2023.
ifeval                       instruction_following  Zhou et al., 2023. IFBench, Allen Institute for AI.
bfcl                         tool_use               Patil, Mao, Yan, Ji, Suresh, Stoica, Gonzalez. BFCL.
tau_bench                    agent                  Yao et al., 2024.
swe_bench                    code                   Jimenez et al., 2024. Chen et al., 2021 for HumanEval pass@k.
ruler                        long_context          